In [133]:
# Cell 0: Config
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
import scipy.stats as stats
import numpy as np
from linearmodels.panel import PanelOLS

SPREADS_PATH = '../output/final/complete_results_weekly_15_dampening.csv'
DATE_COL     = 'date'
COUNTRY_COL  = 'country'
CDS_COL      = 'cds_spread'
YIELD_COL    = 'yield_market'
HORIZON = 5
RECOVERY = 0.4
HORIZONS     = [1, 2, 4, 8]
EXPORTERS    = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Colombia',
                'Mexico', 'Brazil', 'Egypt', 'Malaysia', 'Qatar']
CONTROLS     = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
                'South Korea', 'Thailand', 'Turkey']

panel = pd.read_csv(SPREADS_PATH, parse_dates=[DATE_COL])

In [134]:
# Cell 2: Load controls and merge onto panel

# ── Load macro controls ───────────────────────────────────────
macro = pd.read_csv(
    '../data/processed/Macroeconomic_variables/macro_risk_variables.csv',
    sep=',', dayfirst=True, parse_dates=['Date'], index_col='Date'
)
vix = pd.read_csv(
    '../data/processed/Macroeconomic_variables/VIXCLS.csv',
    parse_dates=['Date'], index_col='Date'
)
ovx = pd.read_csv(
    '../data/processed/Macroeconomic_variables/OVXCLS.csv',
    parse_dates=['date'], index_col='date'
)
gpr = pd.read_csv(
    '../data/processed/Macroeconomic_variables/geopolitical_risk_index_daily.csv',
    sep=';', dayfirst=True, parse_dates=['date'], index_col='date',
    decimal=','
)
for col in gpr.columns:
    gpr[col] = pd.to_numeric(gpr[col], errors='coerce')

oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',
                           parse_dates=['date'], index_col='date')
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv',
                           dayfirst=True, parse_dates=['date'], index_col='date')

# ── Build controls on daily index ────────────────────────────
macro['VIX']   = vix['VIXCLS']
macro['OVX']   = ovx['OVXCLS']
macro          = macro.join(gpr[['GPRD']], how='left')
macro['basis'] = oil_prices['Brent'] / oil_futures['Brent_12m']
controls_daily = macro[['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis']].copy()

# ── Load FX rates ─────────────────────────────────────────────
fx_wide = pd.read_csv('../data/processed/CCA_V2/exchange_rates.csv',
                       dayfirst=True, parse_dates=['date'])
fx = fx_wide.melt(id_vars='date', var_name=COUNTRY_COL, value_name='fx_rate')
fx = fx.dropna(subset=['fx_rate'])

fx[COUNTRY_COL] = fx[COUNTRY_COL].replace({'United Arab Emirates': 'UAE (Abu Dhabi)'})


# ── Anchor to panel dates ─────────────────────────────────────
cds_dates = pd.DatetimeIndex(panel[DATE_COL].unique())

# ── Reindex controls to panel dates ──────────────────────────
controls_weekly = controls_daily\
    .reindex(cds_dates, method='nearest',
             tolerance=pd.Timedelta('7 days'))\
    .ffill().bfill()\
    .reset_index()\
    .rename(columns={'Date': DATE_COL, 'index': DATE_COL})
controls_weekly[DATE_COL] = pd.to_datetime(controls_weekly[DATE_COL])

# ── Reindex FX per country to panel dates ────────────────────
fx_weekly = pd.merge_asof(
    panel[[DATE_COL, COUNTRY_COL]].sort_values(DATE_COL),
    fx.sort_values(DATE_COL),
    on=DATE_COL,
    by=COUNTRY_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

fx_weekly

# ── Merge controls onto panel ─────────────────────────────────
panel = pd.merge_asof(
    panel.sort_values(DATE_COL),
    controls_weekly.sort_values(DATE_COL),
    on=DATE_COL,
    direction='nearest',
    tolerance=pd.Timedelta('7 days')
)

panel = panel.merge(
    fx_weekly[[DATE_COL, COUNTRY_COL, 'fx_rate']],
    on=[DATE_COL, COUNTRY_COL],
    how='left'
)

panel = panel.sort_values([COUNTRY_COL, DATE_COL]).reset_index(drop=True)

print("Panel shape:", panel.shape)
print("Columns:", panel.columns.tolist())
print("Nulls:\n", panel.isnull().sum()[panel.isnull().sum() > 0])


Panel shape: (8352, 33)
Columns: ['date', 'country', 'cds_spread', 'risk_free_rate', 'B_f', 'LCL_usd', 'sigma_lcl', 'implied_V_M0', 'implied_sigma_V_M0', 'cca_converged_M0', 'implied_V_M1', 'implied_sigma_V_M1', 'cca_converged_M1', 'convenience_yield', 'implied_V_M2', 'implied_sigma_V_M2', 'cca_converged_M2', 'lambda_annual', 'group', 'exporter', 'DD_M0', 'DD_M1', 'DD_M2', 'PD_M0', 'PD_M1', 'PD_M2', 'VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD', 'basis', 'fx_rate']
Nulls:
 Series([], dtype: int64)


/var/folders/wy/gjw_3_n51t748hfngpz4zf0w0000gn/T/ipykernel_3615/879418672.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv',


# 0: Correlations

In [135]:
# Correlation of levels: CDS vs DD across models
from scipy.stats import pearsonr, spearmanr

print(f"{'Country':<20} | {'M0 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M1 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8} | {'M2 Pearson':>10} {'p':>8} {'Spearman':>10} {'p':>8}")
print("-" * 130)

corrs = {f'{m}_{g}_{t}': [] for m in ['M0','M1','M2'] for g in ['exp','ctl'] for t in ['pear','spear']}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    sub = panel[panel[COUNTRY_COL] == c].copy()
    
    line = f"{c:<20}"
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        tmp = sub[[CDS_COL, dd_col]].dropna()
        tmp = tmp[tmp[CDS_COL] > 0]
        
        if len(tmp) > 10:
            pr, pp = pearsonr(tmp[CDS_COL], tmp[dd_col])
            sr, sp = spearmanr(tmp[CDS_COL], tmp[dd_col])
        else:
            pr, pp, sr, sp = np.nan, np.nan, np.nan, np.nan
        
        corrs[f'{model}_{grp}_pear'].append(pr)
        corrs[f'{model}_{grp}_spear'].append(sr)
        
        p_star = '***' if pp < 0.01 else '**' if pp < 0.05 else '*' if pp < 0.1 else ''
        s_star = '***' if sp < 0.01 else '**' if sp < 0.05 else '*' if sp < 0.1 else ''
        line += f" | {pr:>9.3f}{p_star:<3} {pp:>8.4f} {sr:>9.3f}{s_star:<3} {sp:>8.4f}"
    
    print(line)

print("-" * 130)
for g, label in [('exp', 'Exporters'), ('ctl', 'Controls')]:
    line = f"Mean {label:<15}"
    for m in ['M0', 'M1', 'M2']:
        mp = np.nanmean(corrs[f'{m}_{g}_pear'])
        ms = np.nanmean(corrs[f'{m}_{g}_spear'])
        line += f" | {mp:>10.3f} {'':>8} {ms:>10.3f} {'':>8}"
    print(line)

Country              | M0 Pearson        p   Spearman        p | M1 Pearson        p   Spearman        p | M2 Pearson        p   Spearman        p
----------------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |    -0.631***   0.0000    -0.765***   0.0000 |    -0.649***   0.0000    -0.800***   0.0000 |    -0.685***   0.0000    -0.781***   0.0000
UAE (Abu Dhabi)      |     0.216***   0.0000     0.127***   0.0036 |    -0.315***   0.0000    -0.261***   0.0000 |    -0.282***   0.0000    -0.330***   0.0000
Colombia             |    -0.574***   0.0000    -0.601***   0.0000 |    -0.533***   0.0000    -0.558***   0.0000 |    -0.599***   0.0000    -0.635***   0.0000
Mexico               |    -0.384***   0.0000    -0.400***   0.0000 |    -0.528***   0.0000    -0.448***   0.0000 |    -0.534***   0.0000    -0.527***   0.0000
Brazil               |    -0.543***   0.0000    -0.566***   0.0000 |    -0.591***   0.

# 1. Inseparability Test: OVX and Futures Basis vs Global Risk Factors

In [136]:
# Use unique dates from panel — one row per date for time series regressions
ts = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)\
          [['VIX', 'OVX', 'DXY', 'UST10Y', 'GPRD']].sort_index()

# ── OVX ~ Global factors (levels) ────────────────────────────
idx   = ts[['OVX', 'VIX', 'DXY', 'UST10Y', 'GPRD']].dropna().index
y_ovx = ts.loc[idx, 'OVX']
X_ovx = sm.add_constant(ts.loc[idx, ['VIX', 'DXY', 'UST10Y', 'GPRD']])
res_ovx = sm.OLS(y_ovx, X_ovx).fit()

print("OVX ~ Global Factors (levels)")
print(f"R²: {res_ovx.rsquared:.4f}  |  Adj. R²: {res_ovx.rsquared_adj:.4f}")
print(f"\n{'Variable':<12} {'Beta':>10} {'p-value':>10}")
print("-" * 35)
for var in X_ovx.columns:
    print(f"{var:<12} {res_ovx.params[var]:>10.4f} {res_ovx.pvalues[var]:>10.4f}")

OVX ~ Global Factors (levels)
R²: 0.5903  |  Adj. R²: 0.5872

Variable           Beta    p-value
-----------------------------------
const         -101.0461     0.0000
VIX              1.4614     0.0000
DXY              1.3520     0.0000
UST10Y          -7.6438     0.0000
GPRD             0.0104     0.3573


In [137]:
# Convenience yield ~ Global Factors
cy = panel.drop_duplicates(subset=[DATE_COL]).set_index(DATE_COL)[['convenience_yield', 'VIX', 'DXY', 'UST10Y', 'GPRD']].sort_index().dropna()

y = cy['convenience_yield']
X = sm.add_constant(cy[['VIX', 'DXY', 'UST10Y', 'GPRD']])
res = sm.OLS(y, X).fit()

print("Convenience Yield ~ Global Factors (levels)")
print(f"R²: {res.rsquared:.4f}  |  Adj. R²: {res.rsquared_adj:.4f}")
print(f"\n{'Variable':<15} {'Beta':>10} {'p-value':>10}")
print("-" * 38)
for var in X.columns:
    print(f"{var:<15} {res.params[var]:>10.4f} {res.pvalues[var]:>10.4f}")

Convenience Yield ~ Global Factors (levels)
R²: 0.2687  |  Adj. R²: 0.2630

Variable              Beta    p-value
--------------------------------------
const               0.1795     0.0742
VIX                -0.0004     0.4985
DXY                -0.0030     0.0100
UST10Y              0.0530     0.0000
GPRD                0.0002     0.0142


# 2. Regressions

In [138]:
# ============================================================
# Regression 1: ln(CDS) = α + β·ln(V) + ε  — Asset Value (M0 vs M1)
# ============================================================

print("Regression 1: ln(CDS) = α + β·ln(V) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)

r2_v = {'M0_exp': [], 'M1_exp': [], 'M0_ctl': [], 'M1_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"

    for model, v_col in [('M0', 'implied_V_M0'), ('M1', 'implied_V_M1')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, v_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[v_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[v_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_v[f'{model}_{grp}'].append(res.rsquared)

    print(line)

print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_v['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_v['M1_ctl']):>7.3f}")

Regression 1: ln(CDS) = α + β·ln(V) + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.576***  0.0000   0.484   522 |  -0.541***  0.0000   0.536   522
UAE (Abu Dhabi)      |  -0.612***  0.0000   0.391   522 |  -0.553***  0.0000   0.445   522
Colombia             |   0.390*    0.0561   0.045   522 |   0.208     0.2060   0.021   522
Mexico               |  -0.848***  0.0000   0.187   522 |  -0.702***  0.0000   0.275   522
Brazil               |  -3.458***  0.0000   0.479   522 |  -1.609***  0.0000   0.421   522
Egypt                |   1.149***  0.0000   0.274   522 |   1.023***  0.0000   0.290   522
Malaysia             |  -2.582***  0.0000   0.515   522 |  -1.887***  0.0000   0.548   522
Qatar                |  -0.815***  0.0000   0.575   522 |  -0.729***  0.0000   0.596   522
Chile                |  -0.235     0.2738   0.016  

In [139]:
# ============================================================
# Regression 2: ln(CDS) = α + β·ln(σ) + ε  — Volatility (M0 vs M2)
# ============================================================
 
print("\n\nRegression 2: ln(CDS) = α + β·ln(σ) + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 90)
 
r2_vol = {'M0_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M2_ctl': []}
 
for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
 
    for model, sig_col in [('M0', 'implied_sigma_V_M0'), ('M2', 'implied_sigma_V_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, sig_col]].dropna()
        df = df[(df[CDS_COL] > 0) & (df[sig_col] > 0)]
        y = np.log(df[CDS_COL])
        X = sm.add_constant(np.log(df[sig_col]))
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        b = res.params.iloc[1]
        p = res.pvalues.iloc[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        r2_vol[f'{model}_{grp}'].append(res.rsquared)
 
    print(line)
 
print("-" * 90)
print(f"{'Mean R² Exporters':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_exp']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_exp']):>7.3f}")
print(f"{'Mean R² Controls':<20} | {'':>8} {'':>8} {np.mean(r2_vol['M0_ctl']):>7.3f} {'':>5} | {'':>8} {'':>8} {np.mean(r2_vol['M2_ctl']):>7.3f}")



Regression 2: ln(CDS) = α + β·ln(σ) + ε

Country              |     M0 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------
Saudi Arabia         |   0.438***  0.0000   0.599   522 |   0.625***  0.0000   0.650   522
UAE (Abu Dhabi)      |  -0.117**   0.0318   0.068   522 |  -0.066     0.5436   0.003   522
Colombia             |   0.677***  0.0014   0.217   522 |   0.821***  0.0007   0.239   522
Mexico               |   0.024     0.8414   0.000   522 |   0.275     0.1442   0.036   522
Brazil               |   0.741***  0.0002   0.194   522 |   1.011***  0.0001   0.257   522
Egypt                |   0.137*    0.0583   0.034   522 |   0.148*    0.0851   0.029   522
Malaysia             |   0.409*    0.0657   0.041   522 |   1.189***  0.0043   0.113   522
Qatar                |   0.388***  0.0000   0.482   522 |   0.624***  0.0000   0.553   522


Chile                |   0.135     0.2900   0.014   522 |   0.457***  0.0060   0.079   522
China                |  -0.543**   0.0408   0.047   522 |  -0.401     0.2296   0.015   522
Indonesia            |  -0.286*    0.0523   0.034   522 |   0.117     0.6910   0.003   522
Philippines          |  -0.364***  0.0000   0.243   522 |  -0.430***  0.0000   0.168   522
South Africa         |   0.216**   0.0143   0.043   522 |   0.525***  0.0004   0.146   522
South Korea          |   0.224     0.3134   0.014   522 |   0.342     0.3011   0.014   522
Thailand             |  -0.497***  0.0019   0.107   522 |  -0.353     0.1694   0.021   522
Turkey               |   0.592***  0.0000   0.435   522 |   0.822***  0.0000   0.511   522
------------------------------------------------------------------------------------------
Mean R² Exporters    |                     0.204       |                     0.235
Mean R² Controls     |                     0.117       |                     0.120


In [140]:
print("Regression: ln(CDS) = α + β·DD + ε\n")
print(f"{'Country':<20} | {'M0 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M1 β':>8} {'p':>8} {'R²':>7} {'N':>5} | {'M2 β':>8} {'p':>8} {'R²':>7} {'N':>5}")
print("-" * 120)

r2 = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}
betas = {'M0_exp': [], 'M1_exp': [], 'M2_exp': [], 'M0_ctl': [], 'M1_ctl': [], 'M2_ctl': []}

for c in EXPORTERS + CONTROLS:
    grp = 'exp' if c in EXPORTERS else 'ctl'
    line = f"{c:<20}"
    
    for model, dd_col in [('M0', 'DD_M0'), ('M1', 'DD_M1'), ('M2', 'DD_M2')]:
        df = panel[panel[COUNTRY_COL] == c][[CDS_COL, dd_col]].dropna()
        df = df[df[CDS_COL] > 0]
        y = np.log(df[CDS_COL].values)
        X = sm.add_constant(df[dd_col].values)
        res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 8})
        
        b = res.params[1]
        p = res.pvalues[1]
        sig = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''
        
        line += f" | {b:>7.3f}{sig:<3} {p:>7.4f} {res.rsquared:>7.3f} {len(df):>5}"
        
        r2[f'{model}_{grp}'].append(res.rsquared)
        betas[f'{model}_{grp}'].append(b)
    
    print(line)

print("-" * 120)

print(
    f"{'Mean β Exporters':<20} | "
    f"{np.mean(betas['M0_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_exp']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_exp']):>7.3f}"
)
print(
    f"{'Mean β Controls':<20} | "
    f"{np.mean(betas['M0_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M1_ctl']):>7.3f} {'':>8} {'':>7} {'':>5} | "
    f"{np.mean(betas['M2_ctl']):>7.3f}"
)
print(
    f"{'Mean R² Exporters':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_exp']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_exp']):>7.3f}"
)
print(
    f"{'Mean R² Controls':<20} | "
    f"{'':>8} {'':>8} {np.mean(r2['M0_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M1_ctl']):>7.3f} {'':>5} | "
    f"{'':>8} {'':>8} {np.mean(r2['M2_ctl']):>7.3f}"
)

Regression: ln(CDS) = α + β·DD + ε

Country              |     M0 β        p      R²     N |     M1 β        p      R²     N |     M2 β        p      R²     N
------------------------------------------------------------------------------------------------------------------------
Saudi Arabia         |  -0.126***  0.0000   0.494   522 |  -0.105***  0.0000   0.514   522 |  -0.283***  0.0000   0.551   522
UAE (Abu Dhabi)      |   0.015     0.2371   0.030   522 |  -0.033**   0.0200   0.089   522 |  -0.175***  0.0094   0.090   522
Colombia             |  -0.259***  0.0000   0.335   522 |  -0.209***  0.0000   0.304   522 |  -0.370***  0.0000   0.354   522
Mexico               |  -0.121**   0.0113   0.139   522 |  -0.107***  0.0001   0.256   522 |  -0.244***  0.0002   0.267   522
Brazil               |  -0.209***  0.0000   0.326   522 |  -0.176***  0.0000   0.370   522 |  -0.310***  0.0000   0.387   522
Egypt                |  -0.069***  0.0000   0.196   522 |  -0.068***  0.0000   0.182   522

## Pooled Regression

## Main Specification: Stacked Panel Regression

For each model extension $M_k \in \{M1, M2\}$. We estimate:

$$\ln(\text{CDS}_{it}) = \alpha_i + \lambda_t + \beta_1 \, DD_{it} + \beta_2 \, (DD_{it} \times \text{Exp}_i) + \beta_3 \, (DD_{it} \times \text{Model}_{it}) + \beta_4 \, (DD_{it} \times \text{Exp}_i \times \text{Model}_{it}) + \varepsilon_{it}$$

with entity and time fixed effects. The coefficients map directly to the three hypotheses:
- **$\beta_1$ (Baseline performance in controls):** How well does DD predict CDS for controls under M0.
- **$\beta_2$ (Baseline differential in exporters):** 
- **$\beta_3$ (H1/H2 for controls):** Does the extension improve DD–CDS tracking for control countries?
- **$\beta_4$ (H3):** Does the improvement differ for exporters relative to controls?
- **$\beta_3 + \beta_4$ (H1/H2 for exporters):** Does the extension improve DD–CDS tracking for exporters? Tested via Wald test of $\beta_3 + \beta_4 = 0$.


In [141]:
# ============================================================
# H3 Test: Does the exporter differential CHANGE across models?
# Triple interaction: DD × Exporter × Model
# ============================================================

from linearmodels.panel import PanelOLS

comparisons = [('M0', 'M1', 'DD_M0', 'DD_M1'),
               ('M0', 'M2', 'DD_M0', 'DD_M2')]

for base_name, ext_name, base_col, ext_col in comparisons:
    # Stack base and extended model
    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()
    d0['DD'] = d0[base_col]
    d0['model'] = 0

    d1 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, ext_col]].dropna().copy()
    d1 = d1[d1[CDS_COL] > 0].copy()
    d1['DD'] = d1[ext_col]
    d1['model'] = 1

    stacked = pd.concat([d0, d1], ignore_index=True)
    stacked['ln_cds']       = np.log(stacked[CDS_COL])
    stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
    stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
    stacked['dd_model']     = stacked['DD'] * stacked['model']
    stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']

    stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

    X = stacked[['DD', 'dd_exp', 'dd_model', 'dd_exp_model']]
    y = stacked['ln_cds']

    mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)
    res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=4)
    #res = mod.fit(cov_type='clustered', cluster_time=True)
    #res = mod.fit(cov_type='clustered', cluster_entity=True)


    g3 = res.params['dd_exp_model']
    p3 = res.pvalues['dd_exp_model']
    sig = '***' if p3 < 0.01 else ('**' if p3 < 0.05 else ('*' if p3 < 0.1 else ''))

    print(f"\n{base_name} → {ext_name}: dd_exp_model = {g3:.4f}  (p = {p3:.4f}) {sig}")
    print(res.summary)
    print(res.wald_test(formula='dd_model + dd_exp_model = 0'))


M0 → M1: dd_exp_model = -0.0025  (p = 0.0873) *
                          PanelOLS Estimation Summary                           
Dep. Variable:                 ln_cds   R-squared:                        0.1093
Estimator:                   PanelOLS   R-squared (Between):             -0.0837
No. Observations:               16704   R-squared (Within):               0.0933
Date:                Mon, Apr 27 2026   R-squared (Overall):             -0.0826
Time:                        20:07:00   Log-likelihood                   -397.28
Cov. Estimator:        Driscoll-Kraay                                           
                                        F-statistic:                      495.84
Entities:                          16   P-value                           0.0000
Avg Obs:                       1044.0   Distribution:                 F(4,16163)
Min Obs:                       1044.0                                           
Max Obs:                       1044.0   F-statistic (robust)

### Same as above but in a reduced summary form

In [142]:
# ============================================================
# Main Specification: Stacked Triple Interaction
# ============================================================
from linearmodels.panel import PanelOLS

comparisons = [('M0', 'M1', 'DD_M0', 'DD_M1'),
               ('M0', 'M2', 'DD_M0', 'DD_M2')]
BANDWIDTHS = [4, 8, 26]

for base_name, ext_name, base_col, ext_col in comparisons:

    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()
    d0['DD'] = d0[base_col]
    d0['model'] = 0

    d1 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, ext_col]].dropna().copy()
    d1 = d1[d1[CDS_COL] > 0].copy()
    d1['DD'] = d1[ext_col]
    d1['model'] = 1

    stacked = pd.concat([d0, d1], ignore_index=True)
    stacked['ln_cds']       = np.log(stacked[CDS_COL])
    stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
    stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
    stacked['dd_model']     = stacked['DD'] * stacked['model']
    stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']
    stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

    X = stacked[['DD', 'dd_exp', 'dd_model', 'dd_exp_model']]
    y = stacked['ln_cds']
    mod = PanelOLS(y, X, entity_effects=True, time_effects=True, drop_absorbed=True)

    print(f"\n{'='*80}")
    print(f"  {base_name} → {ext_name}")
    print(f"{'='*80}")

    header = (f"{'BW':>4} | "
            f"{'β1 (DD)':>10} {'p':>8} | "
            f"{'β2 (DD×Exp)':>14} {'p':>8} | "
            f"{'β3 (DD×Mod)':>14} {'p':>8} | "
            f"{'β4 (DD×Exp×Mod)':>18} {'p':>8} | "
            f"{'β3+β4':>8} {'Wald p':>8}")
    print(header)
    print("-" * len(header))

    for bw in BANDWIDTHS:
        res = mod.fit(cov_type='kernel', kernel='bartlett', bandwidth=bw)
        
        b1 = res.params['DD']
        p1 = res.pvalues['DD']
        b2 = res.params['dd_exp']
        p2 = res.pvalues['dd_exp']
        b3 = res.params['dd_model']
        p3 = res.pvalues['dd_model']
        b4 = res.params['dd_exp_model']
        p4 = res.pvalues['dd_exp_model']
        
        wald = res.wald_test(formula='dd_model + dd_exp_model = 0')
        wp = wald.pval
        
        def stars(p):
            return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        
        print(f"{bw:>4} | "
            f"{b1:>10.4f} {p1:>7.4f}{stars(p1):<3} | "
            f"{b2:>14.4f} {p2:>7.4f}{stars(p2):<3} | "
            f"{b3:>14.4f} {p3:>7.4f}{stars(p3):<3} | "
            f"{b4:>18.4f} {p4:>7.4f}{stars(p4):<3} | "
            f"{b3+b4:>8.4f} {wp:>7.4f}{stars(wp)}")


  M0 → M1
  BW |    β1 (DD)        p |    β2 (DD×Exp)        p |    β3 (DD×Mod)        p |    β4 (DD×Exp×Mod)        p |    β3+β4   Wald p
--------------------------------------------------------------------------------------------------------------------------------
   4 |    -0.0241  0.0000*** |        -0.0336  0.0000*** |         0.0001  0.6122    |            -0.0025  0.0873*   |  -0.0024  0.1414
   8 |    -0.0241  0.0006*** |        -0.0336  0.0002*** |         0.0001  0.6964    |            -0.0025  0.1904    |  -0.0024  0.2619
  26 |    -0.0241  0.0271**  |        -0.0336  0.0159**  |         0.0001  0.8047    |            -0.0025  0.3907    |  -0.0024  0.4705

  M0 → M2
  BW |    β1 (DD)        p |    β2 (DD×Exp)        p |    β3 (DD×Mod)        p |    β4 (DD×Exp×Mod)        p |    β3+β4   Wald p
--------------------------------------------------------------------------------------------------------------------------------
   4 |    -0.0300  0.0000*** |        -0.0299  0.0001*

## Permutation test

In [143]:
np.random.seed(42)
n_perms = 500

placebo_p = {('M0','M1'): [], ('M0','M2'): []}

for base_name, ext_name, base_col, ext_col in comparisons:
    d0 = panel[[COUNTRY_COL, DATE_COL, CDS_COL, base_col, ext_col]].dropna().copy()
    d0 = d0[d0[CDS_COL] > 0].copy()

    for i in range(n_perms):
        # Randomly assign which DD is "model=1" for each row
        swap = np.random.binomial(1, 0.5, size=len(d0)).astype(bool)
        dd_base = np.where(swap, d0[ext_col], d0[base_col])
        dd_ext  = np.where(swap, d0[base_col], d0[ext_col])

        s0 = d0[[COUNTRY_COL, DATE_COL, CDS_COL]].copy()
        s0['DD'] = dd_base
        s0['model'] = 0
        s1 = d0[[COUNTRY_COL, DATE_COL, CDS_COL]].copy()
        s1['DD'] = dd_ext
        s1['model'] = 1

        stacked = pd.concat([s0, s1], ignore_index=True)
        stacked['ln_cds']       = np.log(stacked[CDS_COL])
        stacked['exporter']     = stacked[COUNTRY_COL].isin(EXPORTERS).astype(int)
        stacked['dd_exp']       = stacked['DD'] * stacked['exporter']
        stacked['dd_model']     = stacked['DD'] * stacked['model']
        stacked['dd_exp_model'] = stacked['DD'] * stacked['exporter'] * stacked['model']
        stacked = stacked.set_index([COUNTRY_COL, DATE_COL])

        mod = PanelOLS(stacked['ln_cds'], stacked[['DD','dd_exp','dd_model','dd_exp_model']],
                       entity_effects=True, time_effects=True, drop_absorbed=True)
        res = mod.fit()
        placebo_p[(base_name, ext_name)].append(res.params['dd_exp_model'])

    actual = -0.0025 if ext_name == 'M1' else -0.0236
    placebo = np.array(placebo_p[(base_name, ext_name)])
    perm_p = np.mean(placebo <= actual)
    print(f"{base_name}→{ext_name}: actual={actual:.4f}, placebo mean={placebo.mean():.4f}, "
          f"placebo std={placebo.std():.4f}, permutation p={perm_p:.4f}")

M0→M1: actual=-0.0025, placebo mean=-0.0001, placebo std=0.0006, permutation p=0.0000
M0→M2: actual=-0.0236, placebo mean=0.0000, placebo std=0.0010, permutation p=0.0000
